# PC-RORT interactive research notebook

**One-run workflow:** install the numerical package, reproduce the covariance selection flip, compare uncertainty sets, and visualize statistical power. Use **Run All** in JupyterLab or Google Colab.


In [ ]:
%pip -q install numpy scipy cvxpy pandas matplotlib pytest


In [ ]:
import pathlib, sys, subprocess, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
# In JupyterLab launch the notebook from the repository root.
# In Colab clone the public repository automatically if not already present.
if not pathlib.Path('pcrort/matrix.py').exists():
    subprocess.run(['git','clone','--depth','1','--branch','feature/pcrort-matrix-core','https://github.com/dippendicular/PC-RORT.git','/content/PC-RORT'],check=True)
    import os
    os.chdir('/content/PC-RORT')
sys.path.insert(0,str(pathlib.Path.cwd().resolve()))
from pcrort.matrix import conditional_risk_matrix, relative_contrast
from pcrort.robust import UncertaintySet, robust_select
print('PC-RORT repository:',pathlib.Path.cwd().resolve())


## 1. Covariance-driven source decision
The same marginal variances give different decisions when the common-encoder cross covariance is removed.


In [ ]:
full = np.array([[1.,1.5],[1.5,9.]])
diag = np.diag(np.diag(full))
results=[]
for name,q in [('Full covariance',full),('Diagonal ablation',diag)]:
    fit=robust_select(UncertaintySet(q,'frobenius',0.),rho=.5)
    results.append({'condition':name,'external_weight':round(float(1-fit.weights[0]),6),'contrast':round(float(fit.objective),6)})
display(pd.DataFrame(results))


## 2. PSD uncertainty-set comparison
This cell uses a controlled positive-semidefinite risk-matrix center. Compare solver convergence and nonlocal selection.


In [ ]:
center=np.diag([2.,1.,3.])
sets={'Frobenius':UncertaintySet(center,'frobenius',.05),
      'Entrywise PSD':UncertaintySet(center,'box_psd',widths=np.full((3,3),.05)),
      'Affine PSD':UncertaintySet(center,'affine_psd',.10,directions=np.eye(3)[None,:,:])}
comparison=[]
for name,region in sets.items():
    fit=robust_select(region,rho=.5,max_iterations=60)
    comparison.append({'region':name,'external_mass':1-fit.weights[0],'contrast':fit.objective,'gap':fit.gap,'converged':fit.converged})
display(pd.DataFrame(comparison))


## 3. Monte Carlo coverage and selection power
Choose small values initially. Increasing repetitions and bootstrap draws provides more stable estimates at greater runtime.


In [ ]:
from experiments.run_pcrort_matrix import run
from argparse import Namespace
args=Namespace(samples=[30,60],candidates=[2],dimensions=[1],mismatches=[0.,.3],shared=.5,n_final=50,reps=5,bootstrap=20,seed=20261026,alpha=.05,rho=.5,solver='CLARABEL',max_iterations=35,out='results/notebook_matrix.csv')
path=run(args)
df=pd.read_csv(path)
display(df.head())


In [ ]:
valid=df[df['status']=='ok'].copy()
summary=valid.groupby(['N','mismatch','method'],dropna=False).agg(coverage=('covered','mean'),selection_power=('nonlocal','mean'),mean_true_contrast=('true_contrast','mean'),n=('rep','count')).reset_index()
display(summary)
fig,ax=plt.subplots(figsize=(8,4.5))
for method,group in summary[summary['mismatch']==0.3].groupby('method'):
    ax.plot(group['N'],group['selection_power'],marker='o',label=method)
ax.set(xlabel='Pilot sample size N',ylabel='Nonlocal-selection frequency',title='Source-selection power: controlled mismatch')
ax.set_ylim(-.02,1.02);ax.legend();plt.show()


## Reproduction
For manuscript-grade results, run the CLI with a prespecified grid, many independent repetitions, and preserved CSV/JSON config metadata. Inspect nonconverged solver runs separately from statistical outcomes.

```bash
python -m experiments.run_pcrort_matrix --samples 50 100 250 --candidates 2 5 --dimensions 1 4 16 --mismatches 0 0.3 --reps 200 --bootstrap 500 --out results/paper_matrix.csv
```
